# 01 — Build the Maize/Weed Dataset  ·  run on **CPU**

`Runtime → Change runtime type → CPU`. Nothing here needs a GPU.

**What it makes:** one merged object-detection dataset with two classes — **0 = maize, 1 = weed** (*anything that is not maize is a weed*) — split into **train / val / test**, saved to Drive as `MyDrive/maize_weed/maizeweed_dataset.zip`, plus report-ready figures and tables in `MyDrive/maize_weed/figures/dataset/`.

### Data sources
| key | Dataset | Where | Camera view | Labelled images used | Licence |
|---|---|---|---|---|---|
| `nigeria` | Maize-Weed Image (Olaniyi et al., 2023, *Data in Brief* 47:109030) | Niger State, **Nigeria** | Handheld, ~1 ft above crop | 500 | CC BY 4.0 |
| `cornweed` | CornWeed (DFKI / HS Osnabrück, Zenodo 7961764) | Germany | Vehicle-mounted, top-down | ~3,574 | see Zenodo record |
| `cropandweed` | CropAndWeed (Steininger et al., WACV 2023) — images **containing maize** | Austria | Ground-level, top-down | maize subset of 8,034 | **non-commercial research only** |
| `acre` | ACRE Crop-Weed (Zenodo 8102217) — **maize** images | France (ROSE challenge site) | Robot-mounted, top-down | maize subset of ~1,000 | CC BY 4.0 |
| `rf_cornweed` | corn-weed (Roboflow Universe, emad-yousry) | — | close-up | ~900 | CC BY 4.0 |
| `weedmaize` *(optional, off)* | WeedMaize (López-Correa et al., Zenodo 5106795) | Spain | 1.3 m, top-down | 312 large → ~7.8k tiles | CC BY 4.0 |

### Checked and left out (and why)
- **Roboflow `sabya` maize-weed** — weed boxes drawn around maize plants; labels unreliable.
- **MegaWeeds** (30 GB) — mostly other crops; its maize part *is* WeedMaize.
- **MFWD** (maize & sorghum, TU Munich) — single plants in trays on a conveyor belt, not field scenes.
- **USU-Corn-WeedDB, DRONEWEED, MatchPlant** — drone images; wrong viewpoint for a ground robot.
- **KaraAgro AI Maize (Ghana)**, **Tanzania maize** — pests/diseases, not weeds (useful later for a disease feature).
- **Latvian food-crop & weed set**, **4Weed**, **NDSU weed-crop** — no maize, or crops would be mislabelled as weeds.
- **VERIDIS** (corn & beet robot) — crop labels only; unlabelled weeds would teach the model that weeds are background.
- **Roboflow `faffy` / MaizeWeedDeneme** — no downloadable version / unnamed classes (MaizeWeedDeneme filenames look like the Nigerian set — worth checking later).

**Total download ≈ 26 GB** (to Colab's disk, not your device). Each source is converted and its raw files deleted before the next one.

## Step 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install roboflow
!df -h /content | tail -1   # need ~30 GB free

In [ ]:
import os, shutil
DRIVE_DIR   = '/content/drive/MyDrive/maize_weed'
RAW         = '/content/raw'                      # temporary downloads
OUT         = '/content/maizeweed'                # merged dataset
DATASET_ZIP = f'{DRIVE_DIR}/maizeweed_dataset.zip'
FIG_DIR     = f'{DRIVE_DIR}/figures/dataset'      # report figures & tables
os.makedirs(DRIVE_DIR, exist_ok=True); os.makedirs(RAW, exist_ok=True)

ROBOFLOW_API_KEY = ''   # roboflow.com -> Settings -> API Keys

USE = {                 # which sources to include
    'nigeria':     True,
    'cornweed':    True,
    'cropandweed': True,
    'acre':        True,
    'rf_cornweed': True,
    'weedmaize':   False,   # optional: tiny seedlings, +5.6 GB
}
MAIZE_IMAGES_ONLY = True   # CropAndWeed & ACRE contain other crop fields: keep only images that contain maize
NIGERIA_REPEAT    = 3      # Nigerian TRAINING images are used 3x so the model learns local conditions
VAL_FRACTION      = 0.15
TEST_FRACTION     = 0.15   # held-out test set: only used for the final numbers in your report
MAX_SIDE          = 640

### Start fresh — deletes old dataset zips in Drive and any half-built dataset on this machine

In [ ]:
import glob
for z in glob.glob(f'{DRIVE_DIR}/maizeweed_dataset*.zip'):
    os.remove(z); print('deleted', z)
shutil.rmtree(OUT, ignore_errors=True); shutil.rmtree(RAW, ignore_errors=True); os.makedirs(RAW, exist_ok=True)
print('Clean start.')

## Step 1 — Converters
Every source becomes a 640 px JPG + YOLO label file. Label names are mapped: **maize / corn / zea → maize**, **…row… / soil → dropped**, **anything else → weed**.
Splits are **deterministic** and keep ~10 consecutive frames (or one recording session) together, so near-identical photos never land in both train and test.
After each source a table shows exactly which original label went where.

In [ ]:
import os, re, json, random, shutil, glob, pathlib, hashlib, tarfile, zipfile, xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from PIL import Image
import yaml

IMG_EXTS    = {'.jpg', '.jpeg', '.png', '.bmp'}
CLASS_NAMES = ['maize', 'weed']
DROP_WORDS  = ('row', 'soil')

name_log  = defaultdict(Counter)   # source -> (original label, new class) -> boxes
img_count = Counter()              # (source, split) -> images
_seen, _counter = set(), Counter()

def _cls(name):
    n = str(name).lower()
    if any(w in n for w in DROP_WORDS): return None
    return 0 if any(w in n for w in ('maize', 'corn', 'zea')) else 1

def name_to_cls(name, source):
    cls = _cls(name)
    name_log[source][(str(name), CLASS_NAMES[cls] if cls is not None else 'DROPPED')] += 1
    return cls

def split_for(key, val=None, test=None):
    """Deterministic split from a key (same key -> same split, every run)."""
    val = VAL_FRACTION if val is None else val; test = TEST_FRACTION if test is None else test
    h = int(hashlib.md5(str(key).encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return 'test' if h < test else ('val' if h < test + val else 'train')

def seq_group(path):
    """Group ~10 consecutive frames together (C00020878 -> C0002087) so near-identical frames never
    end up in both train and test."""
    stem = pathlib.Path(path).stem
    return re.sub(r'(\d+)$', lambda m: str(int(m.group(1)) // 10), stem)

def split_from_path(p):
    q = '/' + p.replace('\\', '/').lower() + '/'
    if '/valid/' in q or '/val/' in q: return 'val'
    if '/test/' in q: return 'test'
    if '/train/' in q: return 'train'
    return None

def index_images(root):
    idx = {}
    for p in pathlib.Path(root).rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            idx.setdefault(p.name, str(p)); idx.setdefault(p.stem, str(p))
    return idx

def save_sample(img_path, boxes, source, split=None, group=None, maize_only=False):
    """boxes: list of (cls, cx, cy, w, h) normalised 0-1. Writes a 640px JPG + YOLO label."""
    boxes = [b for b in boxes if b[0] is not None and b[3] > 0 and b[4] > 0]
    if not boxes: return False
    if maize_only and not any(b[0] == 0 for b in boxes): return False
    key = os.path.abspath(img_path)
    if key in _seen: return False
    try: im = Image.open(img_path).convert('RGB')
    except Exception as e:
        print('  skip unreadable', img_path, e); return False
    _seen.add(key)
    split = split or split_for(group or os.path.basename(img_path))
    stem = f'{source}_{_counter[source]:06d}'; _counter[source] += 1
    im.thumbnail((MAX_SIDE, MAX_SIDE))
    os.makedirs(f'{OUT}/images/{split}', exist_ok=True); os.makedirs(f'{OUT}/labels/{split}', exist_ok=True)
    im.save(f'{OUT}/images/{split}/{stem}.jpg', quality=92)
    with open(f'{OUT}/labels/{split}/{stem}.txt', 'w') as f:
        for c, cx, cy, w, h in boxes:
            x0, x1 = max(cx - w / 2, 0), min(cx + w / 2, 1); y0, y1 = max(cy - h / 2, 0), min(cy + h / 2, 1)
            if x1 - x0 <= 0 or y1 - y0 <= 0: continue
            f.write(f'{c} {(x0+x1)/2:.6f} {(y0+y1)/2:.6f} {x1-x0:.6f} {y1-y0:.6f}\n')
    img_count[(source, split)] += 1
    return True

def xyxy_to_box(c, x0, y0, x1, y1, W, H):
    x0, x1 = sorted((x0, x1)); y0, y1 = sorted((y0, y1))
    return (c, (x0 + x1) / 2 / W, (y0 + y1) / 2 / H, (x1 - x0) / W, (y1 - y0) / H)

def find_names(root):
    for p in glob.glob(f'{root}/**/*.yaml', recursive=True):
        try:
            d = yaml.safe_load(open(p))
            if isinstance(d, dict) and 'names' in d:
                n = d['names']
                return {int(k): v for k, v in n.items()} if isinstance(n, dict) else dict(enumerate(n))
        except Exception: pass
    for fn in ('classes.txt', 'obj.names', '_darknet.labels'):
        for p in glob.glob(f'{root}/**/{fn}', recursive=True):
            lines = [l.strip() for l in open(p) if l.strip()]
            if lines: return dict(enumerate(lines))
    return None

# ---------- YOLO folders (boxes or polygons); uses the folder's own train/valid/test split ----------
def convert_yolo_dir(root, names, source, maize_only=False):
    idx = index_images(root); n = 0
    for lp in glob.glob(f'{root}/**/*.txt', recursive=True):
        if os.path.basename(lp).lower().startswith(('classes', 'readme')): continue
        guess = lp.replace('/labels/', '/images/')
        img = next((guess[:-4] + e for e in ('.jpg', '.jpeg', '.png', '.JPG', '.PNG', '.JPEG') if os.path.exists(guess[:-4] + e)), None)
        img = img or idx.get(pathlib.Path(lp).stem)
        if not img: continue
        boxes = []
        for line in open(lp):
            parts = line.split()
            if len(parts) < 5: continue
            try: cid = int(float(parts[0])); v = list(map(float, parts[1:]))
            except ValueError: continue
            if cid not in names: continue
            if len(v) == 4: cx, cy, w, h = v
            elif len(v) >= 6 and len(v) % 2 == 0:
                xs, ys = v[0::2], v[1::2]
                cx, cy, w, h = (min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)
            else: continue
            boxes.append((name_to_cls(names[cid], source), cx, cy, w, h))
        n += save_sample(img, boxes, source, split=split_from_path(lp), maize_only=maize_only)
    return n

# ---------- COCO json ----------
def is_coco(p):
    try:
        if os.path.getsize(p) > 1_000_000_000: return False
        d = json.load(open(p))
        return isinstance(d, dict) and all(k in d for k in ('images', 'annotations', 'categories'))
    except Exception: return False

def convert_coco(json_path, root, source, split_fn=None, maize_only=False):
    d = json.load(open(json_path)); idx = index_images(root); n = 0
    cats = {c['id']: c['name'] for c in d['categories']}
    by_img = defaultdict(list)
    for a in d['annotations']: by_img[a['image_id']].append(a)
    for im in d['images']:
        path = idx.get(os.path.basename(im['file_name'])) or idx.get(pathlib.Path(im['file_name']).stem)
        if not path: continue
        W, H = im['width'], im['height']
        boxes = [xyxy_to_box(name_to_cls(cats[a['category_id']], source), a['bbox'][0], a['bbox'][1],
                             a['bbox'][0] + a['bbox'][2], a['bbox'][1] + a['bbox'][3], W, H) for a in by_img.get(im['id'], [])]
        split = split_fn(seq_group(path)) if split_fn else None
        n += save_sample(path, boxes, source, split=split, group=seq_group(path), maize_only=maize_only)
    return n

# ---------- XML: Pascal VOC, LabelMe-style polygons, or CVAT-for-images ----------
def _num(t):
    try: float(t); return True
    except (TypeError, ValueError): return False

def xml_records(xp):
    """Returns [(image_name, W, H, [(label, x0, y0, x1, y1), ...]), ...] for many XML layouts."""
    try: r = ET.parse(xp).getroot()
    except ET.ParseError: return []
    recs = []
    if r.tag != 'annotation' and r.find('.//image') is not None and r.find('.//image').get('name'):   # CVAT
        for im in r.iter('image'):
            W, H = float(im.get('width') or 0), float(im.get('height') or 0); bx = []
            for el in im:
                lab = el.get('label')
                if el.tag == 'box':
                    bx.append((lab, *(float(el.get(k)) for k in ('xtl', 'ytl', 'xbr', 'ybr'))))
                elif el.get('points'):
                    pts = [tuple(map(float, p.split(','))) for p in el.get('points').split(';') if ',' in p]
                    if pts: xs, ys = zip(*pts); bx.append((lab, min(xs), min(ys), max(xs), max(ys)))
            recs.append((im.get('name'), W, H, bx))
        return recs
    fn = (r.findtext('filename') or '').strip()
    W = float(r.findtext('size/width') or r.findtext('imagesize/ncols') or 0)
    H = float(r.findtext('size/height') or r.findtext('imagesize/nrows') or 0)
    bx = []
    for o in r.iter('object'):
        lab = (o.findtext('name') or o.findtext('label') or '').strip()
        if (o.findtext('deleted') or '0').strip() == '1': continue
        b = o.find('bndbox')
        if b is not None and all(_num(b.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax')):
            bx.append((lab, *(float(b.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax')))); continue
        xs = [float(e.text) for e in o.iter() if re.fullmatch(r'x\d*', e.tag.lower()) and _num(e.text)]
        ys = [float(e.text) for e in o.iter() if re.fullmatch(r'y\d*', e.tag.lower()) and _num(e.text)]
        if xs and ys: bx.append((lab, min(xs), min(ys), max(xs), max(ys)))
    recs.append((fn or pathlib.Path(xp).stem, W, H, bx))
    return recs

def convert_xml(root, source, maize_only=False):
    idx = index_images(root); n = 0
    for xp in glob.glob(f'{root}/**/*.xml', recursive=True):
        for fn, W, H, bx in xml_records(xp):
            path = idx.get(os.path.basename(fn or '')) or idx.get(pathlib.Path(fn or '').stem) or idx.get(pathlib.Path(xp).stem)
            if not path: continue
            if maize_only and not any(_cls(lab) == 0 for lab, *_ in bx): continue
            if not W or not H: W, H = Image.open(path).size
            boxes = [xyxy_to_box(name_to_cls(lab, source), x0, y0, x1, y1, W, H) for lab, x0, y0, x1, y1 in bx]
            n += save_sample(path, boxes, source, group=seq_group(path), maize_only=maize_only)
    if n == 0:
        ex = next(iter(glob.glob(f'{root}/**/*.xml', recursive=True)), None)
        print('No images converted. Example XML (send this to Claude):')
        if ex: print(open(ex).read()[:1500])
    return n

# ---------- CropAndWeed (CSV bboxes: Left,Top,Right,Bottom,LabelID,StemX,StemY) ----------
CNW_MAIZE_IDS = {1, 2, 3, 4, 5, 6}
def parse_cnw_boxes(root):
    out = {}
    csvs = [p for p in glob.glob(f'{root}/**/*.csv', recursive=True) if pathlib.Path(p).parent.name in ('bboxes', 'CropAndWeed')]
    for p in csvs:
        bx = []
        for line in open(p):
            parts = [s.strip() for s in line.split(',')]
            if len(parts) < 5: continue
            try: x0, y0, x1, y1, lid = (float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), int(float(parts[4])))
            except ValueError: continue          # header line
            if lid == 0: continue                 # soil
            c = 0 if lid in CNW_MAIZE_IDS else 1  # everything that is not maize is a weed for our robot
            bx.append((c, x0, y0, x1, y1, lid))
        out[pathlib.Path(p).stem] = bx
    return out

def convert_cropandweed(root, cnw_boxes, keep, source='cropandweed'):
    idx = index_images(root); n = 0
    for stem in keep:
        path = idx.get(stem)
        if not path: continue
        W, H = Image.open(path).size
        boxes = []
        for c, x0, y0, x1, y1, lid in cnw_boxes[stem]:
            lab = 'Maize (ids 1-6)' if c == 0 else ('Vegetation (255)' if lid == 255 else 'Other crops & weeds (7-99)')
            name_log[source][(lab, CLASS_NAMES[c])] += 1
            boxes.append(xyxy_to_box(c, x0, y0, x1, y1, W, H))
        n += save_sample(path, boxes, source, group=stem.rsplit('-', 1)[0])   # group by recording session
    return n

def safe_extract(t, path, members):
    try: t.extractall(path, members=members, filter='data')
    except TypeError: t.extractall(path, members=members)

def show_tree(root, depth=3):
    root = pathlib.Path(root)
    for p in sorted(root.rglob('*')):
        rel = p.relative_to(root)
        if p.is_dir() and len(rel.parts) <= depth:
            print('  ' * (len(rel.parts) - 1) + f'{p.name}/  ({sum(1 for _ in p.iterdir())} items)')

def report(source):
    print(f'\n{source}: original label -> new class (boxes)')
    for (orig, new), c in sorted(name_log[source].items()): print(f'   {orig:<30} -> {new:<8} {c:,}')
    print('   images: ' + ', '.join(f'{s} {img_count[(source, s)]:,}' for s in ('train', 'val', 'test')))

print('Converters ready.')


def save_log():
    os.makedirs(OUT, exist_ok=True)
    json.dump({'name_log': {s: [[o, n, c] for (o, n), c in d.items()] for s, d in name_log.items()},
               'img_count': [[s, sp, c] for (s, sp), c in img_count.items()]}, open(f'{OUT}/build_log.json', 'w'))
_report = report
def report(source):
    _report(source); save_log()

## Step 2 — Download & convert each source

### 2a. Nigeria — Maize-Weed Image (4.2 GB download → 500 labelled images)

In [ ]:
if USE['nigeria']:
    zp, root = f'{RAW}/nigeria.zip', f'{RAW}/nigeria'
    !wget -q --show-progress -O {zp} "https://data.mendeley.com/public-api/zip/jjbfcckrsp/download/2"
    if not zipfile.is_zipfile(zp): raise RuntimeError('Nigeria download failed - tell Claude.')
    with zipfile.ZipFile(zp) as z:
        members = [n for n in z.namelist() if 'annotat' in n.lower()]
        print(len(members), 'files in the annotated folder - extracting only those')
        z.extractall(root, members=members)
    os.remove(zp)
    for inner in glob.glob(f'{root}/**/*.zip', recursive=True):
        with zipfile.ZipFile(inner) as z: z.extractall(inner[:-4])
        os.remove(inner)
    convert_xml(root, 'nigeria'); report('nigeria'); shutil.rmtree(root)

### 2b. CornWeed — Germany (8.7 GB → ~3,574 labelled images). Its official *valid* split becomes our **test** split.

In [ ]:
if USE['cornweed']:
    !wget -q --show-progress -O {RAW}/cornweed.zip "https://zenodo.org/records/7961764/files/maize_dataset.zip?download=1"
    !unzip -q {RAW}/cornweed.zip -d {RAW}/cornweed && rm {RAW}/cornweed.zip
    root = f'{RAW}/cornweed'
    for p in [p for p in glob.glob(f'{root}/**/*.json', recursive=True) if is_coco(p)]:
        is_valid = 'valid' in os.path.basename(p).lower()
        print('COCO file:', p, '-> test' if is_valid else '-> train/val')
        convert_coco(p, root, 'cornweed',
                     split_fn=(lambda k: 'test') if is_valid else (lambda k: 'val' if split_for(k, test=0) == 'val' else 'train'))
    report('cornweed'); shutil.rmtree(root)

### 2c. CropAndWeed — Austria (annotations first, then 4 image archives ≈ 11.6 GB)
Uses the dataset's own **Maize vs everything-else** grouping (label ids 1–6 = maize). Only images that contain maize are kept (`MAIZE_IMAGES_ONLY`), and only those images are extracted from the archives.

In [ ]:
if USE['cropandweed']:
    CNW = 'https://vitro-testing.com/wp-content/uploads/2022/12/'
    root = f'{RAW}/cropandweed'; os.makedirs(root, exist_ok=True)
    tp = f'{RAW}/cnw_annotations.tar'
    if not (os.path.exists(tp) and tarfile.is_tarfile(tp)):
        !wget -q --show-progress -O {tp} "{CNW}cropandweed_annotations.tar"
    if not tarfile.is_tarfile(tp): raise RuntimeError('CropAndWeed annotations download failed - tell Claude.')
    with tarfile.open(tp) as t:
        mem = [m for m in t.getmembers() if m.isfile() and m.name.endswith('.csv') and pathlib.Path(m.name).parent.name in ('bboxes', 'CropAndWeed')]
        if not mem: print([m.name for m in t.getmembers()][:30]); raise RuntimeError('bbox CSVs not found - send the list above to Claude.')
        safe_extract(t, root, mem)
    os.remove(tp)
    cnw_boxes = parse_cnw_boxes(root)
    ids = Counter(lid for bx in cnw_boxes.values() for *_, lid in bx)
    print('label ids (id: boxes):', dict(sorted(ids.items())))
    print('maize boxes (ids 1-6):', sum(v for k, v in ids.items() if k in CNW_MAIZE_IDS))
    keep = {s for s, b in cnw_boxes.items() if any(c == 0 for c, *_ in b)} if MAIZE_IMAGES_ONLY else set(cnw_boxes)
    print(f'{len(cnw_boxes):,} annotated images, {len(keep):,} kept')
    for i in range(1, 5):
        tp = f'{RAW}/cnw_images{i}.tar'
        !wget -q --show-progress -O {tp} "{CNW}cropandweed_images{i}of4.tar"
        if not tarfile.is_tarfile(tp): raise RuntimeError(f'CropAndWeed image archive {i} failed - tell Claude.')
        with tarfile.open(tp) as t:
            mem = [m for m in t.getmembers() if m.isfile() and pathlib.Path(m.name).stem in keep and pathlib.Path(m.name).suffix.lower() in IMG_EXTS]
            safe_extract(t, f'{root}/images', mem); print(f'archive {i}: extracted {len(mem):,} images')
        os.remove(tp)
    convert_cropandweed(root, cnw_boxes, keep); report('cropandweed'); shutil.rmtree(root)

### 2d. ACRE Crop-Weed — France, robot camera (1.2 GB). Maize folders only, converted with ACRE's own script.

In [ ]:
if USE['acre']:
    import types, importlib.util, time
    zp, root = f'{RAW}/acre.zip', f'{RAW}/acre'
    if not os.path.exists(root):
        !wget -q --show-progress -O {zp} "https://zenodo.org/records/8102217/files/The_ACRE_Crop-Weed_Dataset.zip?download=1"
        if not zipfile.is_zipfile(zp): raise RuntimeError('ACRE download failed - tell Claude.')
        with zipfile.ZipFile(zp) as z: z.extractall(root)
        os.remove(zp)
    script = glob.glob(f'{root}/**/acre_xml_to_coco_obj_det.py', recursive=True)[0]
    maize_dir = [p for p in glob.glob(f'{root}/**/Maize', recursive=True) if os.path.isdir(p)][0]
    out_json = f'{RAW}/acre_maize_coco.json'
    spec = importlib.util.spec_from_file_location('acre_conv', script); mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    arg_names = sorted(set(re.findall(r'args\.(\w+)', open(script).read())))
    print('ACRE script arguments:', arg_names)
    t0 = time.time() - 1
    mod.main(types.SimpleNamespace(**{a: (maize_dir if ('dir' in a or 'xml' in a) else out_json) for a in arg_names}))
    # the script chooses its own output location: find the COCO file it just wrote
    new = [p for p in set(glob.glob(f'{root}/**/*.json', recursive=True) + glob.glob(f'{RAW}/*.json') + glob.glob(f'{os.getcwd()}/*.json'))
           if os.path.getmtime(p) >= t0 and is_coco(p)]
    if not new: raise RuntimeError('ACRE script wrote no COCO file - tell Claude.')
    out_json = max(new, key=os.path.getmtime); print('COCO file written by the ACRE script:', out_json)
    d = json.load(open(out_json))
    for c in d['categories']:
        if c['name'].lower() == 'crop': c['name'] = 'maize (ACRE crop)'   # only Maize folders were converted
    json.dump(d, open(out_json, 'w'))
    print(f"{len(d['images'])} images, {len(d['annotations'])} plants in the ACRE maize folders")
    convert_coco(out_json, maize_dir, 'acre'); report('acre'); shutil.rmtree(root)

### 2e. Roboflow corn-weed (small). Uses the project's own train/valid/test split.

In [ ]:
if USE['rf_cornweed']:
    from roboflow import Roboflow
    assert ROBOFLOW_API_KEY, 'Paste your Roboflow API key in the settings cell and re-run it.'
    p = Roboflow(api_key=ROBOFLOW_API_KEY).workspace('emad-yousry').project('corn-weed')
    v = max(p.versions(), key=lambda v: int(str(v.version).split('/')[-1]))
    ds = v.download('yolov8', location=f'{RAW}/rf_cornweed', overwrite=True)
    names = find_names(ds.location); print('names:', names)
    convert_yolo_dir(ds.location, names, 'rf_cornweed'); report('rf_cornweed'); shutil.rmtree(ds.location)

### 2f. (Optional) WeedMaize — Spain (5.6 GB). Off unless `USE['weedmaize'] = True`.

In [ ]:
if USE['weedmaize']:
    root = f'{RAW}/weedmaize'
    for part in ('train_set', 'test_set', 'validation_set'):
        !wget -q --show-progress -O {RAW}/{part}.zip "https://zenodo.org/record/5106795/files/{part}.zip?download=1"
        !unzip -q {RAW}/{part}.zip -d {root}/{part} && rm {RAW}/{part}.zip
    convert_xml(root, 'weedmaize'); report('weedmaize'); shutil.rmtree(root)
else:
    print('WeedMaize skipped.')

## Step 3 — Clean & balance (safe to re-run)
Removes sources switched off in `USE`, then copies the Nigerian **training** images `NIGERIA_REPEAT` times. Validation and test are never copied, so scores stay honest.

In [ ]:
def src_of(path): return os.path.basename(path).rsplit('_', 1)[0]
removed = made = 0
for split in ('train', 'val', 'test'):
    for ip in glob.glob(f'{OUT}/images/{split}/*.jpg'):
        s = src_of(ip)
        if s.startswith('nigeriarep') or USE.get(s) is False:
            os.remove(ip); removed += 1
            lp = ip.replace('/images/', '/labels/')[:-4] + '.txt'
            if os.path.exists(lp): os.remove(lp)
for k in range(1, NIGERIA_REPEAT):
    for ip in glob.glob(f'{OUT}/images/train/nigeria_*.jpg'):
        stem = os.path.basename(ip)[:-4].replace('nigeria_', f'nigeriarep{k}_', 1)
        shutil.copy(ip, f'{OUT}/images/train/{stem}.jpg')
        shutil.copy(ip.replace('/images/', '/labels/')[:-4] + '.txt', f'{OUT}/labels/train/{stem}.txt'); made += 1
print(f'removed {removed} | made {made} extra Nigerian training copies')
with open(f'{OUT}/data.yaml', 'w') as f:
    f.write(f'path: {OUT}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: maize\n  1: weed\n')

## Step 4 — Statistics, figures & tables for your report
Everything is saved to `MyDrive/maize_weed/figures/dataset/` as **300-dpi PNG + vector PDF** (figures) and **CSV + LaTeX** (tables). Copies of Nigerian images (`nigeriarep`) are left out of the statistics.

In [ ]:
# ---------- Figure style (used for every report figure) ----------
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
INK, INK2, GRID, SURFACE = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
CLS_COLOR = {'maize': '#008300', 'weed': '#4a3aa7'}           # validated colour-blind-safe pair
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']  # fixed order
BOX_COLOR = {0: (0, 200, 83), 1: (255, 64, 214)}             # bright overlays on photos: maize green, weed magenta
SEQ_CMAP = LinearSegmentedColormap.from_list('seq_blue', ['#fcfcfb', '#cde2fb', '#86b6ef', '#2a78d6', '#184f95', '#0d366b'])
mpl.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'savefig.facecolor': SURFACE, 'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.titlecolor': INK, 'axes.labelcolor': INK2, 'axes.edgecolor': GRID, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'axes.grid.axis': 'y', 'xtick.color': INK2, 'ytick.color': INK2, 'legend.frameon': False,
    'lines.linewidth': 2, 'font.family': 'DejaVu Sans'})

def savefig(fig, name, show=True):
    os.makedirs(FIG_DIR, exist_ok=True)
    for ext in ('png', 'pdf'):
        fig.savefig(f'{FIG_DIR}/{name}.{ext}', bbox_inches='tight')
    print(f'saved {FIG_DIR}/{name}.png/.pdf')
    if show: plt.show()
    plt.close(fig)

def label_bars(ax, bars, fmt='{:,.0f}'):
    for b in bars:
        v = b.get_height()
        if v <= 0: continue
        ax.annotate(fmt.format(v), (b.get_x() + b.get_width() / 2, v), xytext=(0, 3),
                    textcoords='offset points', ha='center', va='bottom', fontsize=8, color=INK2)

def save_table(df, name):
    os.makedirs(FIG_DIR, exist_ok=True)
    df.to_csv(f'{FIG_DIR}/{name}.csv', index=False)
    try:
        with open(f'{FIG_DIR}/{name}.tex', 'w') as f: f.write(df.to_latex(index=False, float_format='%.3f'))
    except Exception: pass
    print(f'saved {FIG_DIR}/{name}.csv/.tex')

from PIL import ImageDraw
def draw_boxes(im, boxes, width=None, colors=None):
    """boxes: list of (cls, x0, y0, x1, y1) in pixels. Returns a copy with boxes drawn."""
    im = im.copy(); d = ImageDraw.Draw(im); width = width or max(2, round(max(im.size) / 250))
    for c, x0, y0, x1, y1 in boxes:
        d.rectangle([x0, y0, x1, y1], outline=(colors or BOX_COLOR)[int(c)], width=width)
    return im

def read_label(img_path):
    im = Image.open(img_path); W, H = im.size; out = []
    lp = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if os.path.exists(lp):
        for line in open(lp):
            c, cx, cy, w, h = map(float, line.split())
            out.append((int(c), (cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H))
    return im, out

from matplotlib.patches import Patch
BOX_LEGEND = [Patch(facecolor='none', edgecolor=tuple(v/255 for v in BOX_COLOR[0]), linewidth=2, label='maize'),
              Patch(facecolor='none', edgecolor=tuple(v/255 for v in BOX_COLOR[1]), linewidth=2, label='weed')]

In [ ]:
import pandas as pd, numpy as np
from collections import Counter
SRC_ORDER = ['nigeria', 'cornweed', 'cropandweed', 'acre', 'rf_cornweed', 'weedmaize']
SRC_LABEL = {'nigeria': 'Nigeria (Maize-Weed Image)', 'cornweed': 'CornWeed (Germany)', 'cropandweed': 'CropAndWeed (Austria)',
             'acre': 'ACRE (France)', 'rf_cornweed': 'Roboflow corn-weed', 'weedmaize': 'WeedMaize (Spain)'}
SRC_SHORT = {'nigeria': 'Nigeria', 'cornweed': 'CornWeed', 'cropandweed': 'CropAndWeed', 'acre': 'ACRE',
             'rf_cornweed': 'Roboflow\ncorn-weed', 'weedmaize': 'WeedMaize'}
SRC_INFO = {  # for the dataset table
    'nigeria':     ('Niger State, Nigeria', 'handheld, ~0.3 m', 'CC BY 4.0', '10.17632/jjbfcckrsp.2'),
    'cornweed':    ('Germany', 'vehicle, top-down', 'see record', '10.5281/zenodo.7961764'),
    'cropandweed': ('Austria', 'ground, top-down', 'non-commercial', 'WACV 2023 (Steininger et al.)'),
    'acre':        ('France', 'robot, top-down', 'CC BY 4.0', '10.5281/zenodo.8102217'),
    'rf_cornweed': ('-', 'close-up', 'CC BY 4.0', 'universe.roboflow.com/emad-yousry/corn-weed'),
    'weedmaize':   ('Spain', '1.3 m, top-down', 'CC BY 4.0', '10.5281/zenodo.5106795')}

rows, imgs = [], []
for split in ('train', 'val', 'test'):
    for lp in glob.glob(f'{OUT}/labels/{split}/*.txt'):
        s = src_of(lp)
        if s.startswith('nigeriarep'): continue
        W, H = Image.open(lp.replace('/labels/', '/images/')[:-4] + '.jpg').size
        cnt = Counter()
        for line in open(lp):
            c, cx, cy, w, h = line.split(); c = int(c); cnt[c] += 1
            rows.append((s, split, CLASS_NAMES[c], float(cx), float(cy), float(w), float(h), W, H))
        imgs.append((s, split, cnt[0], cnt[1], W, H))
boxes  = pd.DataFrame(rows, columns=['source', 'split', 'cls', 'cx', 'cy', 'w', 'h', 'W', 'H'])
images = pd.DataFrame(imgs, columns=['source', 'split', 'n_maize', 'n_weed', 'W', 'H'])
boxes['size_px'] = np.sqrt(boxes.w * boxes.W * boxes.h * boxes.H) * 640 / boxes[['W', 'H']].max(axis=1)  # size at 640-px training input
SOURCES = [s for s in SRC_ORDER if s in set(images.source)]
SRC_COLOR = {s: SERIES[i] for i, s in enumerate(SRC_ORDER)}     # colour follows the source, never its rank
n_rep = len(glob.glob(f'{OUT}/images/train/nigeriarep*.jpg'))

# ---- Table 1: dataset summary ----
t = []
for s in SOURCES:
    im, bx = images[images.source == s], boxes[boxes.source == s]
    t.append({'source': SRC_LABEL[s], 'location': SRC_INFO[s][0], 'camera': SRC_INFO[s][1],
              'train': int((im.split == 'train').sum()), 'val': int((im.split == 'val').sum()), 'test': int((im.split == 'test').sum()),
              'maize_boxes': int((bx.cls == 'maize').sum()), 'weed_boxes': int((bx.cls == 'weed').sum()),
              'licence': SRC_INFO[s][2], 'reference': SRC_INFO[s][3]})
summary = pd.DataFrame(t)
tot = summary[['train', 'val', 'test', 'maize_boxes', 'weed_boxes']].sum()
summary = pd.concat([summary, pd.DataFrame([{'source': 'TOTAL', **tot.to_dict()}])], ignore_index=True).fillna('')
save_table(summary, 'table1_dataset_summary')
print(f'(+ {n_rep} extra copies of Nigerian training images used during training)')
display(summary)

# ---- Table 2: label mapping ----
log = json.load(open(f'{OUT}/build_log.json'))['name_log']
mapping = pd.DataFrame([{'source': SRC_LABEL.get(s, s), 'original_label': o, 'mapped_to': n, 'boxes': c}
                        for s in SRC_ORDER if s in log for o, n, c in log[s]])
save_table(mapping, 'table2_label_mapping'); display(mapping)

In [ ]:
# Fig 1 - images per source and split
fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(SOURCES)); wd = 0.26
for i, sp in enumerate(['train', 'val', 'test']):
    vals = [int(((images.source == s) & (images.split == sp)).sum()) for s in SOURCES]
    label_bars(ax, ax.bar(x + (i - 1) * wd, vals, wd, color=SERIES[i], label=sp, edgecolor=SURFACE, linewidth=1))
ax.set_xticks(x, [SRC_LABEL[s].replace(' (', '\n(').replace('Roboflow ', 'Roboflow\n') for s in SOURCES], fontsize=8.5)
ax.set_ylabel('images'); ax.set_title('Images per source and split'); ax.legend(ncol=3, loc='upper right')
savefig(fig, 'fig1_images_per_source_split')

In [ ]:
# Fig 2 - annotated plants per class: totals (log scale) and maize share
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={'width_ratios': [1.4, 1]})
x = np.arange(len(SOURCES)); wd = 0.38
for i, c in enumerate(CLASS_NAMES):
    vals = [int(((boxes.source == s) & (boxes.cls == c)).sum()) for s in SOURCES]
    label_bars(a, a.bar(x + (i - .5) * wd, vals, wd, color=CLS_COLOR[c], label=c, edgecolor=SURFACE, linewidth=1))
a.set_yscale('log'); a.set_xticks(x, [SRC_SHORT[s] for s in SOURCES], fontsize=8.5)
a.set_ylabel('annotated plants (log scale)'); a.set_title('Annotated plants per class'); a.legend()
share = [100 * ((boxes.source == s) & (boxes.cls == 'maize')).sum() / max((boxes.source == s).sum(), 1) for s in SOURCES]
y = np.arange(len(SOURCES))
b.barh(y, share, color=CLS_COLOR['maize'], edgecolor=SURFACE, label='maize')
b.barh(y, [100 - v for v in share], left=share, color=CLS_COLOR['weed'], edgecolor=SURFACE, label='weed')
for yi, v in zip(y, share): b.text(v + 1, yi, f'{v:.0f}% maize', va='center', fontsize=8, color='white' if v < 85 else INK)
b.set_yticks(y, [SRC_SHORT[s] for s in SOURCES], fontsize=8.5); b.invert_yaxis()
b.set_xlim(0, 100); b.set_xlabel('% of annotated plants'); b.set_title('Class balance'); b.grid(False); b.grid(axis='x')
savefig(fig, 'fig2_class_balance')

In [ ]:
# Fig 3 - plants per image (spread per source)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=False)
for ax, c, col in zip(axes, CLASS_NAMES, ['n_maize', 'n_weed']):
    data = [images.loc[images.source == s, col].values for s in SOURCES]
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, widths=0.55, medianprops=dict(color=INK))
    for p in bp['boxes']: p.set_facecolor(CLS_COLOR[c]); p.set_alpha(0.75); p.set_edgecolor(CLS_COLOR[c])
    ax.set_xticks(range(1, len(SOURCES) + 1), [SRC_SHORT[s] for s in SOURCES], fontsize=8.5)
    ax.set_ylabel(f'{c} plants per image'); ax.set_title(f'{c.capitalize()} plants per image')
savefig(fig, 'fig3_plants_per_image')

In [ ]:
# Fig 4 - object size at the 640-px training input (COCO small < 32 px <= medium < 96 px <= large)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
bins = np.logspace(np.log10(2), np.log10(640), 50)
for ax, c in zip(axes, CLASS_NAMES):
    for s in SOURCES:
        v = boxes.loc[(boxes.source == s) & (boxes.cls == c), 'size_px']
        if len(v): ax.hist(v, bins=bins, histtype='step', linewidth=1.8, density=True, color=SRC_COLOR[s], label=SRC_LABEL[s].split(' (')[0])
    for xv in (32, 96):
        ax.axvline(xv, color=INK2, linestyle='--', linewidth=1)
        ax.text(xv * 1.06, 0.97, f'{xv} px', transform=ax.get_xaxis_transform(), fontsize=7.5, color=INK2, va='top')
    for xv, lab in ((10, 'small'), (55, 'medium'), (250, 'large')):
        ax.text(xv, 1.01, lab, transform=ax.get_xaxis_transform(), fontsize=8, color=INK2, ha='center', va='bottom')
    ax.set_xscale('log'); ax.set_xlabel('object size (px, sqrt(width x height) at 640-px input)'); ax.set_ylabel('density')
    ax.set_title(f'{c.capitalize()} object size', pad=16)
axes[1].legend(fontsize=7.5)
savefig(fig, 'fig4_object_size')
buckets = pd.cut(boxes.size_px, [0, 32, 96, 1e9], labels=['small (<32 px)', 'medium (32-96 px)', 'large (>=96 px)'])
size_tab = (pd.crosstab(boxes.cls, buckets, normalize='index') * 100).round(1).reset_index()
save_table(size_tab, 'table3_object_size_buckets_percent'); display(size_tab)

In [ ]:
# Fig 5 - where objects appear in the image (centre heatmaps)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), layout='constrained')
for ax, c in zip(axes, CLASS_NAMES):
    v = boxes[boxes.cls == c]
    h = ax.hist2d(v.cx, v.cy, bins=40, range=[[0, 1], [0, 1]], cmap=SEQ_CMAP)
    ax.invert_yaxis(); ax.set_aspect('equal'); ax.grid(False)
    ax.set_xlabel('x (fraction of width)'); ax.set_ylabel('y (fraction of height)'); ax.set_title(f'{c.capitalize()} centre positions')
    fig.colorbar(h[3], ax=ax, shrink=0.85, label='objects')
savefig(fig, 'fig5_spatial_heatmap')

In [ ]:
# Fig 6 - example annotated images, one row per source (maize = green, weed = magenta)
random.seed(1)
fig, axes = plt.subplots(len(SOURCES), 3, figsize=(12, 3.1 * len(SOURCES)), squeeze=False)
for row, s in zip(axes, SOURCES):
    files = random.sample(glob.glob(f'{OUT}/images/*/{s}_*.jpg'), 3)
    for ax, fp in zip(row, files):
        im, bx = read_label(fp); ax.imshow(draw_boxes(im, bx)); ax.axis('off')
    row[0].set_title(SRC_LABEL[s], loc='left', fontsize=10)
fig.legend(handles=BOX_LEGEND, loc='upper right', ncol=2)
plt.tight_layout(); savefig(fig, 'fig6_examples_per_source')

# Fig 7 - Nigerian examples (local conditions)
files = random.sample(glob.glob(f'{OUT}/images/*/nigeria_*.jpg'), 6)
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, fp in zip(axes.flat, files):
    im, bx = read_label(fp); ax.imshow(draw_boxes(im, bx)); ax.axis('off')
fig.legend(handles=BOX_LEGEND, loc='upper right', ncol=2); fig.suptitle('Annotated Nigerian maize-field images', fontweight='bold')
plt.tight_layout(); savefig(fig, 'fig7_nigeria_examples')

In [ ]:
# Suggested captions for your report
captions = {
 'fig1_images_per_source_split': 'Number of images per data source in the training, validation and test splits.',
 'fig2_class_balance': 'Left: annotated maize and weed plants per source (log scale). Right: share of maize among annotated plants.',
 'fig3_plants_per_image': 'Distribution of the number of maize (left) and weed (right) plants per image for each source (boxes: interquartile range; line: median).',
 'fig4_object_size': 'Object size distribution at the 640-pixel network input. Dashed lines mark the COCO small/medium/large thresholds (32 and 96 px).',
 'fig5_spatial_heatmap': 'Spatial distribution of maize (left) and weed (right) object centres across all images.',
 'fig6_examples_per_source': 'Example images from each data source with ground-truth boxes (green: maize, magenta: weed).',
 'fig7_nigeria_examples': 'Example annotated images from the Nigerian Maize-Weed Image dataset (Niger State).',
 'table1_dataset_summary': 'Data sources, splits and number of annotated plants per class.',
 'table2_label_mapping': 'Mapping of each source\'s original labels to the two classes used in this work.',
 'table3_object_size_buckets_percent': 'Percentage of small, medium and large objects per class.'}
with open(f'{FIG_DIR}/CAPTIONS.md', 'w') as f:
    for k, v in captions.items(): f.write(f'**{k}** — {v}\n\n')
print(open(f'{FIG_DIR}/CAPTIONS.md').read())

## Step 5 — Save the dataset to Drive

In [ ]:
!cd /content && rm -f "{DATASET_ZIP}" && zip -qr "{DATASET_ZIP}" maizeweed
!ls -lh "{DATASET_ZIP}"
print('Done. Switch to a T4 GPU runtime and open 02_train_export.')